# 칩위스퍼러 플랫폼 설정 & 사전 작업 & 바이너리 컴파일 및 프로그램

In [ ]:
PLATFORM = 'CWHUSKY'
#PLATFORM = 'CW308_STM32F3'

In [ ]:
%run '../base/My_Setup.ipynb'

# Husky 및 타겟 보드 테스트 1: assert(골든모델 == CW출력) 

- 예제 코드 : 128바이트를 초과하는 데이터의 처리 방안 예시 

In [ ]:
%run '1.1.Test_1.ipynb'

# Husky 및 타겟 보드 테스트 2: 암호연산 샘플 사이즈(트리거 길이) 설정

In [ ]:
MAX_DATA_LEN = 245  # 245이하 크기 설정
NUM_PRESAMPLES = 0 # 트리거 전 프리 샘플 사이즈
random.seed(1)

data_k = bytearray(random.randint(0, 255) for _ in range(MAX_DATA_LEN))
data_p = bytearray(random.randint(0, 255) for _ in range(MAX_DATA_LEN))

In [ ]:
%run '1.2.Test_2.ipynb'

# Husky 및 타겟 보드 테스트 3: 파형 수집 테스트

In [ ]:
MAX_DATA_LEN = 245  # 245이하 크기 설정
NUM_PRESAMPLES = 0 # 트리거 전 프리 샘플 사이즈
random.seed(1)

data_k = bytearray(random.randint(0, 255) for _ in range(MAX_DATA_LEN))
data_p = bytearray(random.randint(0, 255) for _ in range(MAX_DATA_LEN))

In [ ]:
%run '1.3.Test_3.ipynb'

# Husky 및 타겟 보드 파형 수집 데모

- 스코프 초기화 및 타겟보드 리셋

In [ ]:
scope.default_setup()

# 타겟보드 리셋 함수 reset_target(scope)  ->  ../base/Setup_Generic.ipynb  
reset_target(scope)
time.sleep(1)
reset_target(scope)
time.sleep(1)

- 사전 작업

In [ ]:
MAX_DATA_LEN = 16 
NUM_PRESAMPLES = 0


# init
my_fsr_cmd(target, 0x81, 'k', bytearray(MAX_DATA_LEN))
my_fsr_cmd(target, 0x81, 'p', bytearray(MAX_DATA_LEN))
my_fsr_cmd(target, 0x81, 'l', bytearray([MAX_DATA_LEN]))

# 샘플 개수 설정
MAX_TR_LEN = my_setting_num_samples(target, scope, NUM_PRESAMPLES)  

- 다수의 파형 수집

In [ ]:
debug_mode = True   #True False

NUM_OF_TRACES = 10 
FileDir = '../traces'
TV_case = 'fixed'    # fixed  random 


if debug_mode :
    NUM_OF_TRACES = 5    
    array_i_k = []
    array_i_p = []
    array_o = []
    array_t = []

with h5py.File(f'{FileDir}/tmp_SCA_DB.h5', 'w') as f: 
    dset_i_k = f.create_dataset('i_k', shape=(0, MAX_DATA_LEN), \
                              maxshape=(None, MAX_DATA_LEN), dtype='uint8', chunks=True)
    dset_i_p = f.create_dataset('i_p', shape=(0, MAX_DATA_LEN), \
                              maxshape=(None, MAX_DATA_LEN), dtype='uint8', chunks=True)


    dset_o = f.create_dataset('o', shape=(0, MAX_DATA_LEN), \
                              maxshape=(None, MAX_DATA_LEN), dtype='uint8', chunks=True)

    dset_t = f.create_dataset('t', shape=(0, MAX_TR_LEN), \
                              maxshape=(None, MAX_TR_LEN), dtype='float32', chunks=True)

    
    for i in trange(NUM_OF_TRACES, desc='Capturing traces'):
    
        if TV_case == 'fixed':
            data_i_k = bytearray(b'\xA0' * (MAX_DATA_LEN)) 
            data_i_p = bytearray(b'\x08' * (MAX_DATA_LEN))       
        elif TV_case == 'random':
            data_i_k = bytearray([random.randint(0,255) for _ in range(MAX_DATA_LEN)])  
            data_i_p = bytearray([random.randint(0,255) for _ in range(MAX_DATA_LEN)])       
        else :
            print('TV_case is invalid')
            break
        
        # init
        if (
            data_i_k != my_fsr_cmd(target, 0x81, 'k', data_i_k, payload_only=True) or
            data_i_p != my_fsr_cmd(target, 0x81, 'p', data_i_p, payload_only=True) or
            MAX_DATA_LEN != my_fsr_cmd(target, 0x81, 'l', bytearray([MAX_DATA_LEN]), payload_only=True)[0]
        ):
            print('init fail!')
            continue

        # 파형 수집
        try:
            data_o, trace = my_get_trace(target, scope)
        except (TimeoutError, ValueError) as e:
            print(e) 
            continue                   
                
        if debug_mode :         
            print(f'---- for {i} ----')      
            print(f'input  : {data_i_k.hex(" ")} \n')   
            print(f'input  : {data_i_p.hex(" ")} \n')          
            print(f'output : {data_o.hex(" ")} \n')

            array_i_k.append(data_i_k)
            array_i_p.append(data_i_p)
            array_o.append(data_o)  
            array_t.append(trace)
            
        else :
            dset_i_k.resize(dset_i_k.shape[0] + 1, axis=0)
            dset_i_p.resize(dset_i_p.shape[0] + 1, axis=0)
            dset_o.resize(dset_o.shape[0] + 1, axis=0)
            dset_t.resize(dset_t.shape[0] + 1, axis=0)
            
            dset_i_k[-1:] = np.array(data_i_k)
            dset_i_p[-1:] = np.array(data_i_p)
            dset_o[-1:] = np.array(data_o)
            dset_t[-1:] = np.array(trace)
    
    
if debug_mode :        
    %matplotlib inline    
    fig = plt.plot(np.transpose(np.array(array_t)))  
else :
    d = time.strftime('%Y-%m-%d-%H-%M-%S-', time.localtime(time.time()))
    file = Path(f'{FileDir}/tmp_SCA_DB.h5')
    file.rename(f'{FileDir}/{d}_SCA_DB.h5')

print('Done!\n\n')

In [ ]:
scope.dis()
target.dis()